
## Objective

Compare campaign performance across `campaign_type` for the 2025 campaign
population (see notebook 01-03), to help Marketing understand which
campaign types consistently produce better results.

### Grain
One row per campaign_type (7 types expected).

In [0]:
%sql
SELECT
    COUNT(*) AS campaign_count,
    COUNT(DISTINCT campaign_type) AS distinct_types
FROM `campaign&promotion`.gold.campaign_performance;

In [0]:
%sql

CREATE OR REPLACE VIEW `campaign&promotion`.gold.campaign_type_summary AS

WITH campaign_level AS (
    SELECT
        *,
        ROUND(transaction_value / NULLIF(budget, 0), 2) AS value_to_budget_ratio
    FROM `campaign&promotion`.gold.campaign_performance
)

SELECT
    campaign_type,
    COUNT(*) AS campaign_count,

    SUM(budget) AS total_budget,
    SUM(reward_cost) AS total_reward_cost,
    ROUND(
        100.0 * SUM(reward_cost) / NULLIF(SUM(budget), 0),
        2
    ) AS budget_utilization_pct,

    SUM(eligible_customers) AS total_eligible_customers,
    SUM(participating_customers) AS total_participating_customers,
    ROUND(
        100.0 * SUM(participating_customers) / NULLIF(SUM(eligible_customers), 0),
        2
    ) AS weighted_participation_rate_pct,
    ROUND(AVG(participation_rate_pct), 2) AS avg_participation_rate_pct,

    SUM(transaction_count) AS total_transaction_count,
    SUM(transaction_value) AS total_transaction_value,
    ROUND(
        SUM(transaction_value) / NULLIF(SUM(transaction_count), 0),
        2
    ) AS avg_transaction_value,

    SUM(new_customers) AS total_new_customers,
    ROUND(
        SUM(reward_cost) / NULLIF(SUM(new_customers), 0),
        2
    ) AS cac_reward_basis_weighted,
    ROUND(
        SUM(budget) / NULLIF(SUM(new_customers), 0),
        2
    ) AS cac_budget_basis_weighted,

    
    ROUND(
        SUM(transaction_value) / NULLIF(SUM(reward_cost), 0),
        2
    ) AS value_to_reward_cost_weighted,
    ROUND(AVG(transaction_value_to_reward_cost), 2) AS value_to_reward_cost_avg_per_campaign,

    ROUND(
        SUM(transaction_value) / NULLIF(SUM(budget), 0),
        2
    ) AS value_to_budget_weighted,
    ROUND(AVG(value_to_budget_ratio), 2) AS value_to_budget_avg_per_campaign

FROM campaign_level
GROUP BY campaign_type;

In [0]:
%sql
SELECT
    SUM(campaign_count) AS campaigns_in_summary,
    (SELECT COUNT(*) FROM `campaign&promotion`.gold.campaign_performance) AS campaigns_in_source,
    SUM(total_transaction_value) AS transaction_value_in_summary,
    (SELECT SUM(transaction_value) FROM `campaign&promotion`.gold.campaign_performance) AS transaction_value_in_source
FROM `campaign&promotion`.gold.campaign_type_summary;

In [0]:
%sql
-- Cost Efficiency By Type
SELECT
    campaign_type,
    campaign_count,
    total_transaction_value,
    total_reward_cost,
    total_budget,
    value_to_reward_cost_weighted,
    value_to_budget_weighted
FROM `campaign&promotion`.gold.campaign_type_summary
ORDER BY value_to_reward_cost_weighted DESC;

In [0]:
%sql
-- Weighted Vs Simple Average Gap
SELECT
    campaign_type,
    campaign_count,
    value_to_reward_cost_weighted,
    value_to_reward_cost_avg_per_campaign,
    ROUND(
        value_to_reward_cost_avg_per_campaign - value_to_reward_cost_weighted,
        2
    ) AS avg_vs_weighted_gap
FROM `campaign&promotion`.gold.campaign_type_summary
ORDER BY ABS(value_to_reward_cost_avg_per_campaign - value_to_reward_cost_weighted) DESC;

In [0]:
%sql
-- Participation and Acquistion by type
SELECT
    campaign_type,
    campaign_count,
    total_eligible_customers,
    total_participating_customers,
    weighted_participation_rate_pct,
    avg_participation_rate_pct,
    total_new_customers,
    cac_reward_basis_weighted, -- Customer Acquisition Cost based on reward/promotion cost
    cac_budget_basis_weighted -- Customer Acquisition Cost based on the campaign budget
FROM `campaign&promotion`.gold.campaign_type_summary
ORDER BY total_new_customers DESC;

In [0]:
%sql
-- Budget discipline by type
SELECT
    campaign_type,
    campaign_count,
    total_budget,
    total_reward_cost,
    budget_utilization_pct
FROM `campaign&promotion`.gold.campaign_type_summary
ORDER BY budget_utilization_pct DESC;